# Phase 4 — OCR avec prétraitement d'image

**Objectif** : évaluer si un prétraitement d'image (avant OCR) améliore la
reconnaissance de texte, par rapport à la baseline OCR brut de la Phase 3.

**Prétraitements implémentés** :
- Correction d'inclinaison (deskew)
- Débruitage (denoising)
- Amélioration du contraste (CLAHE)
- Redimensionnement (upscaling)
- Binarisation adaptative

> Ce notebook est **autonome** : toutes les fonctions nécessaires (prétraitement,
> OCR, métriques) sont définies directement ici, sans dépendre du contenu de
> `src/`. Il détecte aussi automatiquement où se trouvent le dataset et les
> images dégradées de la Phase 2, quelle que soit leur convention de nommage.


In [ ]:
import json
import platform
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytesseract
from jiwer import cer, wer

# Configuration automatique de Tesseract sous Windows
if platform.system() == "Windows":
    for _path in [r"C:\Program Files\Tesseract-OCR\tesseract.exe",
                  r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"]:
        if os.path.exists(_path):
            pytesseract.pytesseract.tesseract_cmd = _path
            break

HERE = Path.cwd()
print("Dossier courant (cwd) :", HERE.resolve())


## 0. Détection automatique du dataset et des images dégradées (Phase 2)

In [ ]:
def find_split_dir(candidates_roots, split_names):
    """Cherche un dossier contenant images/ + annotations/ parmi plusieurs noms possibles."""
    for root in candidates_roots:
        for name in split_names:
            candidate = root / name
            if (candidate / "images").is_dir() and (candidate / "annotations").is_dir():
                return candidate
    return None


def find_project_root(start, markers=("src", "dataset", "data"), max_levels=8):
    """Remonte les dossiers parents à la recherche d'un repère du projet."""
    current = start
    for _ in range(max_levels):
        if any((current / m).is_dir() for m in markers):
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


PROJECT_ROOT = find_project_root(HERE)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Impossible de trouver la racine du projet depuis {HERE}.")
print("Racine du projet détectée :", PROJECT_ROOT.resolve())

POSSIBLE_ROOTS = [PROJECT_ROOT, PROJECT_ROOT / "dataset", PROJECT_ROOT / "data", HERE, HERE.parent]
POSSIBLE_ROOTS = [r for r in POSSIBLE_ROOTS if r.exists()]

RAW_DIR = find_split_dir(POSSIBLE_ROOTS, ["training_data", "raw"])
if RAW_DIR is None:
    raise FileNotFoundError("Dossier d'entraînement (training_data/raw) introuvable.")
print("Dossier train détecté :", RAW_DIR.resolve())


In [ ]:
def find_degraded_dir(dataset_root):
    """Cherche un dossier nommé 'degraded' à un ou deux niveaux sous dataset_root."""
    candidates = [dataset_root / "degraded"]
    for child in dataset_root.iterdir() if dataset_root.is_dir() else []:
        if child.is_dir():
            candidates.append(child / "degraded")
    for c in candidates:
        if c.is_dir():
            return c
    return None


def find_manifest_file(degraded_dir):
    """Cherche le fichier manifeste JSON de la Phase 2, quel que soit son nom exact."""
    if degraded_dir is None:
        return None
    json_files = list(degraded_dir.glob("*.json"))
    # priorité aux noms explicites
    for preferred in ["degradation_log.json", "manifest_degraded.json"]:
        for f in json_files:
            if f.name == preferred:
                return f
    # sinon, n'importe quel json contenant "degrad" dans le nom
    for f in json_files:
        if "degrad" in f.name.lower() or "manifest" in f.name.lower():
            return f
    return json_files[0] if json_files else None


DATASET_ROOT = RAW_DIR.parent  # dossier parent de training_data/raw (ex: dataset/ ou data/)
DEGRADED_DIR = find_degraded_dir(DATASET_ROOT)
MANIFEST_FILE = find_manifest_file(DEGRADED_DIR)

print("Dossier dégradées détecté :", DEGRADED_DIR.resolve() if DEGRADED_DIR else "AUCUN")
print("Fichier manifeste détecté :", MANIFEST_FILE.resolve() if MANIFEST_FILE else "AUCUN")

if MANIFEST_FILE is None:
    raise FileNotFoundError(
        "Aucun manifeste de dégradation trouvé. Assure-toi d'avoir exécuté le notebook "
        "02_degradation_generation.ipynb avant celui-ci."
    )


## 1. Normalisation du manifeste de dégradation

Le schéma exact des champs (`source_image` vs `document_original`, `level_name` vs
`level`, présence ou non d'un champ `filename`...) peut varier selon la version du
notebook 02 utilisée. Cette cellule normalise tout vers un format commun, et localise
le fichier image réel sur le disque (qu'il soit dans `degraded/` directement ou dans
un sous-dossier `degraded/images/`).

In [ ]:
raw_records = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
print(f"{len(raw_records)} entrées brutes dans le manifeste")
print("Exemple d'entrée brute :", raw_records[0])


def get_first(d, keys, default=None):
    for k in keys:
        if k in d and d[k]:
            return d[k]
    return default


def locate_image_file(filename, degraded_dir):
    """Cherche le fichier image dans degraded/ ou degraded/images/ (ou sous-dossiers)."""
    direct = degraded_dir / filename
    if direct.exists():
        return direct
    nested = degraded_dir / "images" / filename
    if nested.exists():
        return nested
    matches = list(degraded_dir.rglob(filename))
    return matches[0] if matches else None


normalized = []
for r in raw_records:
    source_image = get_first(r, ["source_image", "document_original", "document", "source"])
    degradation_name = get_first(r, ["degradation"])
    level = get_first(r, ["level_name", "level"])
    output_name = get_first(r, ["output_image", "filename", "image", "path"])

    if output_name is None and source_image and degradation_name and level:
        # reconstruction par convention de nommage si le champ est absent
        stem = Path(source_image).stem
        output_name = f"{stem}__{degradation_name}__{level}.png"

    img_path = locate_image_file(Path(output_name).name, DEGRADED_DIR) if output_name else None

    if source_image and img_path is not None:
        normalized.append({
            "source_image": source_image,
            "degradation": degradation_name,
            "level_name": level,
            "image_path": img_path,
        })

df_norm = pd.DataFrame(normalized)
print(f"{len(df_norm)} entrées normalisées et localisées sur le disque (sur {len(raw_records)})")
df_norm.head(5)


## 2. Fonctions de prétraitement d'image (autonomes)

Chaque fonction est indépendante et composable via `preprocess_pipeline`.

In [ ]:
def to_gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img


def deskew(img):
    gray = to_gray(img)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 10:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) > 15:
        return img
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255))


def denoise(img):
    if len(img.shape) == 3:
        return cv2.fastNlMeansDenoisingColored(img, None, 8, 8, 7, 21)
    return cv2.fastNlMeansDenoising(img, None, 8, 7, 21)


def enhance_contrast(img):
    gray = to_gray(img)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    return clahe.apply(gray)


def resize(img, scale=1.5):
    h, w = img.shape[:2]
    return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)


def adaptive_binarize(img):
    gray = to_gray(img)
    return cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)


STEPS = {"deskew": deskew, "denoise": denoise, "contrast": enhance_contrast,
         "resize": resize, "binarize": adaptive_binarize}


def preprocess_pipeline(img, steps=("deskew", "denoise", "contrast", "resize", "binarize")):
    out = img.copy()
    applied = []
    for step in steps:
        out = STEPS[step](out)
        applied.append(step)
    return out, applied


## 3. Fonctions OCR + métriques (autonomes)

In [ ]:
def run_ocr(img, lang="eng"):
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return pytesseract.image_to_string(rgb, lang=lang).strip()


def ground_truth_text(annotation):
    return " ".join(item["text"] for item in annotation["form"])


def ocr_metrics(hypothesis, reference):
    reference, hypothesis = reference.strip(), hypothesis.strip()
    if not reference:
        return {"CER": None, "WER": None}
    return {
        "CER": round(cer(reference, hypothesis) if hypothesis else 1.0, 4),
        "WER": round(wer(reference, hypothesis) if hypothesis else 1.0, 4),
    }


## 4. Comparaison OCR brut vs OCR prétraité

Sur un sous-échantillon (limité pour un temps d'exécution raisonnable — Tesseract
prend ~1.5-3s par image, et chaque document est traité deux fois : brut + prétraité).

In [ ]:
N_COMPARE = 20
subset = df_norm.head(N_COMPARE).copy()
print(f"{len(subset)} versions dégradées utilisées pour cette comparaison")

records = []
for _, row in subset.iterrows():
    ann_path = RAW_DIR / "annotations" / f"{Path(row['source_image']).stem}.json"
    if not ann_path.exists():
        continue
    ann = json.loads(ann_path.read_text(encoding="utf-8"))
    ref_text = ground_truth_text(ann)

    img = cv2.imread(str(row["image_path"]))
    if img is None:
        continue

    raw_text = run_ocr(img)
    raw_metrics = ocr_metrics(raw_text, ref_text)

    pre_img, steps_applied = preprocess_pipeline(img)
    pre_text = run_ocr(pre_img)
    pre_metrics = ocr_metrics(pre_text, ref_text)

    records.append({
        "document": row["source_image"], "degradation": row["degradation"], "level": row["level_name"],
        "CER_brut": raw_metrics["CER"], "CER_pretraite": pre_metrics["CER"],
        "WER_brut": raw_metrics["WER"], "WER_pretraite": pre_metrics["WER"],
    })

df_compare = pd.DataFrame(records)
df_compare


In [ ]:
print("Moyennes :")
means = df_compare[["CER_brut", "CER_pretraite", "WER_brut", "WER_pretraite"]].mean().round(3)
print(means)

gain_cer = means["CER_brut"] - means["CER_pretraite"]
gain_wer = means["WER_brut"] - means["WER_pretraite"]
print(f"\nGain moyen CER : {gain_cer:+.3f}  (positif = amélioration)")
print(f"Gain moyen WER : {gain_wer:+.3f}  (positif = amélioration)")


## 5. Visualisation : CER brut vs prétraité, par dégradation

In [ ]:
summary = df_compare.groupby("degradation")[["CER_brut", "CER_pretraite"]].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(summary))
width = 0.35
ax.bar(x - width/2, summary["CER_brut"], width, label="OCR brut", color="#A64B42")
ax.bar(x + width/2, summary["CER_pretraite"], width, label="OCR prétraité", color="#5B7B6B")
ax.set_xticks(x)
ax.set_xticklabels(summary["degradation"], rotation=30, ha="right")
ax.set_ylabel("CER")
ax.set_title("CER : OCR brut vs OCR prétraité, par dégradation")
ax.legend()
plt.tight_layout()

fig_dir = PROJECT_ROOT / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "phase4_brut_vs_pretraite.png", dpi=110)
plt.show()


## 6. Quel prétraitement individuel apporte le plus ?

On teste chaque étape du pipeline isolément (au lieu de toutes les combiner), sur un
petit sous-échantillon, afin d'identifier laquelle contribue le plus au gain observé.

In [ ]:
N_ABLATION = 8
ablation_subset = df_norm.head(N_ABLATION).copy()

individual_steps = ["deskew", "denoise", "contrast", "resize", "binarize"]
ablation_records = []

for _, row in ablation_subset.iterrows():
    ann_path = RAW_DIR / "annotations" / f"{Path(row['source_image']).stem}.json"
    if not ann_path.exists():
        continue
    ann = json.loads(ann_path.read_text(encoding="utf-8"))
    ref_text = ground_truth_text(ann)
    img = cv2.imread(str(row["image_path"]))
    if img is None:
        continue

    for step in individual_steps:
        pre_img, _ = preprocess_pipeline(img, steps=(step,))
        text = run_ocr(pre_img)
        metrics = ocr_metrics(text, ref_text)
        ablation_records.append({"step": step, "CER": metrics["CER"]})

df_ablation = pd.DataFrame(ablation_records)
ablation_summary = df_ablation.groupby("step")["CER"].mean().sort_values().reset_index()

results_dir = PROJECT_ROOT / "results" / "tables"
results_dir.mkdir(parents=True, exist_ok=True)
ablation_summary.to_csv(results_dir / "phase4_ablation_pretraitements.csv", index=False)
ablation_summary


## 7. Résultats attendus de la Phase 4 — récapitulatif

- **Pipeline de prétraitement** : deskew, débruitage, amélioration du contraste,
  redimensionnement, binarisation adaptative — chaque étape testée individuellement
  et en combinaison.
- **Comparaison OCR brut vs prétraité** : voir tableau § 4 et graphique § 5
  (`results/figures/phase4_brut_vs_pretraite.png`).
- **Étude d'ablation** : `results/tables/phase4_ablation_pretraitements.csv` — identifie
  quelle étape individuelle contribue le plus à la réduction du CER.
- **Limite à documenter** : le gain du prétraitement dépend fortement du type de
  dégradation (un prétraitement optimisé pour le bruit n'est pas forcément optimal
  pour le flou ou la rotation) — cohérent avec la recommandation du cahier des charges
  de proposer "les meilleurs prétraitements selon les cas".
- **Prochaine étape (Phase 5)** : extraction d'entités et exploitation du layout,
  à partir des sorties OCR (brutes ou prétraitées selon ce qui fonctionne le mieux).
